# Shared S3 Filesystem with AgentCore Runtime

This notebook demonstrates how multiple AgentCore Runtime sessions can share a common filesystem backed by S3.

**What you'll learn:**
- How to give every agent session access to a shared S3 bucket via a Claude skill
- How files written in one session are instantly visible in another
- The contrast between per-session isolated storage (`/mnt/workspace`) and shared storage (S3)

**Architecture:**
```
Session A ──writes──► shared_file_manager.py ──put──► S3 shared/
                                                            │
Session B ──reads──► shared_file_manager.py ──get──► S3 shared/
```

The agent uses the **Claude Agent SDK** and a `shared-files` skill that syncs directly to S3 on every read/write — no polling delay.

### Tutorial Details

| Information         | Details                                              |
|:--------------------|:-----------------------------------------------------|
| Tutorial type       | Shared persistent filesystem across sessions         |
| Agent framework     | Claude Agent SDK                                     |
| Tutorial components | AgentCore Runtime, S3, Claude skill                  |
| Example complexity  | Medium                                               |
| SDK used            | `bedrock-agentcore`, `claude-agent-sdk`, `boto3`     |

## Step 1: Install Dependencies

In [ ]:
!uv pip install -qU -r requirements.txt

In [ ]:
!uv pip freeze | grep boto3

**Restart your kernel before continuing.**

## Step 2: Setup — S3 Bucket + IAM Role

We need:
1. An S3 bucket for the shared filesystem (one bucket, shared across all sessions)
2. An IAM execution role for the AgentCore Runtime, with permissions to read/write that bucket

In [ ]:
import boto3
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f'Region:     {region}')
print(f'Account ID: {account_id}')

### Create the shared S3 bucket

In [ ]:
BUCKET_NAME = f'agentcore-shared-fs-{account_id}'

s3 = boto3.client('s3', region_name=region)

try:
    if region == 'us-east-1':
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={'LocationConstraint': region},
        )
    print(f'✅ Created bucket: {BUCKET_NAME}')
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f'ℹ️ Bucket already exists: {BUCKET_NAME}')

BUCKET_ARN = f'arn:aws:s3:::{BUCKET_NAME}'
print(f'Bucket ARN: {BUCKET_ARN}')

### Create the IAM execution role

The role includes standard AgentCore permissions plus `s3:GetObject`, `s3:PutObject`, `s3:DeleteObject`, and `s3:ListBucket` on the shared bucket.

In [ ]:
import sys
sys.path.insert(0, 'helpers')
from utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME, BUCKET_ARN)
print(f'\nRole ARN: {execution_role_arn}')

## Step 3: Build & Push Docker Image

The agent image contains:
- `main.py` — BedrockAgentCoreApp + Claude Agent SDK entrypoint
- `.claude/skills/shared-files/` — the skill that syncs files to S3

Requires Docker and ECR access.

In [ ]:
import base64
import subprocess

REPO_NAME = 'agentcore-shared-fs-agent'

ecr = boto3.client('ecr', region_name=region)

try:
    resp = ecr.create_repository(repositoryName=REPO_NAME)
    repo_uri = resp['repository']['repositoryUri']
    print(f'✅ Created ECR repository: {repo_uri}')
except ecr.exceptions.RepositoryAlreadyExistsException:
    resp = ecr.describe_repositories(repositoryNames=[REPO_NAME])
    repo_uri = resp['repositories'][0]['repositoryUri']
    print(f'ℹ️ Repository already exists: {repo_uri}')

In [ ]:
auth = ecr.get_authorization_token()
token = base64.b64decode(
    auth['authorizationData'][0]['authorizationToken']
).decode().split(':')[1]

registry = f'{account_id}.dkr.ecr.{region}.amazonaws.com'
subprocess.run(
    f'echo {token} | docker login --username AWS --password-stdin {registry}',
    shell=True, check=True
)

In [ ]:
subprocess.run(['docker', 'build', '-t', f'{REPO_NAME}:latest', 'agent/'], check=True)

In [ ]:
subprocess.run(['docker', 'tag', f'{REPO_NAME}:latest', f'{repo_uri}:latest'], check=True)
subprocess.run(['docker', 'push', f'{repo_uri}:latest'], check=True)
print(f'✅ Pushed: {repo_uri}:latest')

## Step 4: Create AgentCore Runtime

We configure the runtime with `filesystemConfigurations: [{sessionStorage: {mountPath: "/mnt/workspace"}}]` — each AgentCore session gets its own isolated `/mnt/workspace` for private state (Claude SDK session files).

The shared S3 bucket is derived automatically inside the agent from the AWS account ID, using the same naming convention used when the bucket was created above.

The contrast is intentional: `/mnt/workspace` is **per-session**, S3 is **shared across all sessions**.

In [ ]:
import time

acc = boto3.client('bedrock-agentcore-control', region_name=region)

RUNTIME_NAME = 'shared_fs_agent'

response = acc.create_agent_runtime(
    agentRuntimeName=RUNTIME_NAME,
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': f'{repo_uri}:latest',
        }
    },
    roleArn=execution_role_arn,
    protocolConfiguration={'serverProtocol': 'HTTP'},
    networkConfiguration={'networkMode': 'PUBLIC'},
    filesystemConfigurations=[{
        'sessionStorage': {
            'mountPath': '/mnt/workspace'
        }
    }]
)

agent_arn = response['agentRuntimeArn']
agent_id  = response['agentRuntimeId']
print(f'Agent ARN: {agent_arn}')
print(f'Agent ID:  {agent_id}')

In [ ]:
# Uncomment to update an existing runtime instead of creating a new one
# response = acc.update_agent_runtime(
#     agentRuntimeId=agent_id,
#     agentRuntimeArtifact={
#         'containerConfiguration': {
#             'containerUri': f'{repo_uri}:latest',
#         }
#     },
#     roleArn=execution_role_arn,
#     protocolConfiguration={'serverProtocol': 'HTTP'},
#     networkConfiguration={'networkMode': 'PUBLIC'},
#     filesystemConfigurations=[{
#         'sessionStorage': {'mountPath': '/mnt/workspace'}
#     }]
# )

In [ ]:
# Wait for the runtime to become READY
for _ in range(30):
    status = acc.get_agent_runtime(agentRuntimeId=agent_id)['status']
    print(f'Status: {status}')
    if status == 'READY':
        break
    time.sleep(10)

## Step 5: Demo — Shared Filesystem Across Sessions

We'll invoke the agent twice with **no shared `runtimeSessionId`** — two completely independent sessions running on separate containers.

- **Session A** writes a shared file
- **Session B** reads that file without ever having seen it before

This works because both sessions go through S3, not local disk.

In [ ]:
import json

agentcore = boto3.client('bedrock-agentcore', region_name=region)


def invoke(prompt: str, session_id: str | None = None) -> tuple[str, str]:
    """Invoke the agent, stream the response, return (text, session_id)."""
    kwargs = {
        'agentRuntimeArn': agent_arn,
        'qualifier': 'DEFAULT',
        'payload': json.dumps({'prompt': prompt}),
    }
    if session_id:
        kwargs['runtimeSessionId'] = session_id

    response = agentcore.invoke_agent_runtime(**kwargs)
    returned_session_id = response.get('runtimeSessionId', '')

    text = ''
    for raw_line in response['response'].iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue
        try:
            data = json.loads(line[6:])
        except json.JSONDecodeError:
            continue
        if 'content' in data and 'model' in data:
            for item in data['content']:
                if 'text' in item:
                    text += item['text']

    return text.strip(), returned_session_id

### Session A: Write a shared file

In [ ]:
response_a, session_id_a = invoke(
    "Write a shared file called team-notes.txt with the following content: "
    "'Q2 planning meeting notes: launch target is June 15th, budget approved.'"
)

print('Session A ID:', session_id_a)
print()
print(response_a)

#### Verify the file landed in S3

In [ ]:
obj = s3.get_object(Bucket=BUCKET_NAME, Key='shared/team-notes.txt')
print('S3 contents:', obj['Body'].read().decode())

### Session B: Read from a completely different session

No `session_id` is passed — this is a brand-new session with no local state.
The agent has never seen `team-notes.txt` on disk, but the skill reads from S3.

In [ ]:
response_b, session_id_b = invoke("What shared files are available? Read team-notes.txt for me.")

print('Session B ID:', session_id_b)
print('(different from Session A):', session_id_a != session_id_b)
print()
print(response_b)

### Session B: Append to the file

Session B can also update the shared file — Session A (or any future session) will immediately see the update.

In [ ]:
response_c, _ = invoke(
    "Update team-notes.txt — append a new line: 'Action item: Alice to send calendar invites by EOD.'",
    session_id=session_id_b,
)
print(response_c)

#### Confirm the update is in S3 and visible to Session A

In [ ]:
# Check S3 directly
obj = s3.get_object(Bucket=BUCKET_NAME, Key='shared/team-notes.txt')
print('S3 (updated):', obj['Body'].read().decode())
print()

# Session A can see the update from Session B
response_d, _ = invoke(
    "Read the latest team-notes.txt",
    session_id=session_id_a,
)
print('Session A sees:', response_d)

### Inspect containers with `invoke_agent_runtime_command`

We can shell into a live session container to verify:
- `/tmp/shared/` contains the local cache written by the skill
- `/mnt/workspace/` contains only Session B's private state (no cross-contamination)

In [ ]:
def exec_command(session_id: str, command: str) -> str:
    response = agentcore.invoke_agent_runtime_command(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=session_id,
        body={'command': command, 'timeout': 30},
    )
    stdout = ''
    for event in response['stream']:
        if 'chunk' not in event:
            continue
        chunk = event['chunk']
        if 'contentDelta' in chunk:
            stdout += chunk['contentDelta'].get('stdout', '')
    return stdout


print('=== Session B /tmp/shared/ ===')
print(exec_command(session_id_b, 'ls -la /tmp/shared/'))

print('=== Session B /mnt/workspace/ (private, session-isolated) ===')
print(exec_command(session_id_b, 'ls -la /mnt/workspace/'))

## Step 6: Clean Up

Delete the runtime, S3 bucket contents + bucket, ECR repository, and IAM role.

In [ ]:
acc.delete_agent_runtime(agentRuntimeId=agent_id)
print(f'✅ Deleted runtime: {agent_id}')

In [ ]:
# Empty and delete the S3 bucket
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET_NAME):
    objects = [{'Key': obj['Key']} for obj in page.get('Contents', [])]
    if objects:
        s3.delete_objects(Bucket=BUCKET_NAME, Delete={'Objects': objects})

s3.delete_bucket(Bucket=BUCKET_NAME)
print(f'✅ Deleted bucket: {BUCKET_NAME}')

In [ ]:
ecr.delete_repository(repositoryName=REPO_NAME, force=True)
print(f'✅ Deleted ECR repository: {REPO_NAME}')

In [ ]:
from utils import delete_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME
delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)